Import libraries

In [100]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [101]:
%time train_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Women/train_women.csv")
%time test_df =  pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Women/test_women.csv")
%time val_df = pd.read_csv(r"/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/dataset/Hillstrom/Women/val_women.csv")

CPU times: user 28.6 ms, sys: 6 ms, total: 34.6 ms
Wall time: 33.5 ms
CPU times: user 13.1 ms, sys: 999 μs, total: 14.1 ms
Wall time: 14.1 ms
CPU times: user 4.86 ms, sys: 0 ns, total: 4.86 ms
Wall time: 4.86 ms


In [102]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['spend']
treatment_feature = ['treatment']

In [103]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [104]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.78953321  1.6460815   1.41745802 -1.11692492  0.91314538  1.07643872
   0.9950542  -0.36894675 -0.88924565  1.13133938]]


In [105]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [106]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25615, 10]); y=torch.Size([25615, 1]); t=torch.Size([25615, 1])
Shape of val: x=torch.Size([4270, 10]); y=torch.Size([4270, 1]); t=torch.Size([4270, 1])
Shape of test: x=torch.Size([12808, 10]); y=torch.Size([12808, 1]); t=torch.Size([12808, 1])


In [107]:
epochs = 150
alpha = 0.1
lr = 1e-4
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 30
shared_hidden = 200
outcome_hidden = 100
outcome_dropout = 0
shared_dropout = 0.0
activation = torch.nn.ReLU

print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 0.1
 learning rate = 0.0001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [108]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfr = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=lr, 
        alpha = alpha, method=method,
        weight_decay=wd, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        outcome_dropout=outcome_dropout, shared_dropout=shared_dropout,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfr.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfr.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 31
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini


Epoch 1/150 | Loss: -0.7936 | Val Loss: 192.8730 | Val Qini: 0.7880 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7880 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 466.0265 | Val Loss: 192.8251 | Val Qini: 0.7708 (patience: 1/20)EMA Trend: 0.7854 | (patience: 1/20)
Epoch 3/150 | Loss: -0.7801 | Val Loss: 192.7720 | Val Qini: 0.7576 (patience: 2/20)EMA Trend: 0.7812 | (patience: 2/20)
Epoch 4/150 | Loss: -0.7787 | Val Loss: 192.7174 | Val Qini: 0.5595 (patience: 3/20)EMA Trend: 0.7480 | (patience: 3/20)
Epoch 5/150 | Loss: -0.7656 | Val Loss: 192.6570 | Val Qini: 0.4421 (patience: 4/20)EMA Trend: 0.7021 | (patience: 4/20)
Epoch 6/150 | Loss: -0.7610 | Val Loss: 192.5909 | Val Qini: 0.3973 (patience: 5/20)EMA Trend: 0.6564 | (patience: 5/20)
Epoch 7/150 | Loss: -0.7300 | Val Loss: 192.5207 | Val Qini: 0.3818 (patience: 6/20)EMA Trend: 0.6152 | (patience: 6/20)
Epoch 8/150 | Loss: -0.7193 | Val Loss: 192.4484 | Val Qini: 0.3733 (patience: 7/20)EMA Trend: 0.5789 | (patience: 7/20)
Epoch 9/150 |

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7890 | Val Loss: 192.8292 | Val Qini: -0.1421 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1421 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.7876 | Val Loss: 192.7940 | Val Qini: -0.0610 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1299 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: -0.7886 | Val Loss: 192.7573 | Val Qini: -0.0229 ⭐ NEW BEST (peak ≥ trend)EMA Trend: -0.1139 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: -0.7736 | Val Loss: 192.7170 | Val Qini: -0.0682 ✓ above trend but not peak (patience: 1/20)EMA Trend: -0.1070 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: -0.7705 | Val Loss: 192.6723 | Val Qini: -0.1337 (patience: 2/20)EMA Trend: -0.1110 | (patience: 2/20)
Epoch 6/150 | Loss: 274.8282 | Val Loss: 192.6218 | Val Qini: -0.2822 (patience: 3/20)EMA Trend: -0.1367 | (patience: 3/20)
Epoch 7/150 | Loss: -0.7358 | Val Loss: 192.5614 | Val Qini: -0.2987 (patience: 4/20)EMA Trend: -0.1610 | (patience: 4/20)
Epoch 8/150 | Loss: -0.7052 | 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


...

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7934 | Val Loss: 192.8468 | Val Qini: 1.4652 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.4652 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.7854 | Val Loss: 192.8078 | Val Qini: 1.4213 (patience: 1/20)EMA Trend: 1.4586 | (patience: 1/20)
Epoch 3/150 | Loss: -0.7759 | Val Loss: 192.7705 | Val Qini: 1.2925 (patience: 2/20)EMA Trend: 1.4337 | (patience: 2/20)
Epoch 4/150 | Loss: -0.7720 | Val Loss: 192.7346 | Val Qini: 0.9552 (patience: 3/20)EMA Trend: 1.3619 | (patience: 3/20)
Epoch 5/150 | Loss: -0.7713 | Val Loss: 192.6961 | Val Qini: 0.8387 (patience: 4/20)EMA Trend: 1.2834 | (patience: 4/20)
Epoch 6/150 | Loss: -0.7596 | Val Loss: 192.6525 | Val Qini: 0.9738 (patience: 5/20)EMA Trend: 1.2370 | (patience: 5/20)
Epoch 7/150 | Loss: -0.7536 | Val Loss: 192.6048 | Val Qini: 1.0183 (patience: 6/20)EMA Trend: 1.2042 | (patience: 6/20)
Epoch 8/150 | Loss: -0.7286 | Val Loss: 192.5533 | Val Qini: 1.0173 (patience: 7/20)EMA Trend: 1.1761 | (patience: 7/20)
Epoch 9/150 | 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -0.7996 | Val Loss: 192.8915 | Val Qini: 1.0785 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0785 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -0.7988 | Val Loss: 192.8580 | Val Qini: 1.1131 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 1.0837 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: -0.7976 | Val Loss: 192.8225 | Val Qini: 1.0651 (patience: 1/20)EMA Trend: 1.0809 | (patience: 1/20)
Epoch 4/150 | Loss: -0.7795 | Val Loss: 192.7835 | Val Qini: 0.9985 (patience: 2/20)EMA Trend: 1.0685 | (patience: 2/20)
Epoch 5/150 | Loss: -0.7870 | Val Loss: 192.7392 | Val Qini: 1.0151 (patience: 3/20)EMA Trend: 1.0605 | (patience: 3/20)
Epoch 6/150 | Loss: -0.7635 | Val Loss: 192.6874 | Val Qini: 1.0468 (patience: 4/20)EMA Trend: 1.0585 | (patience: 4/20)
Epoch 7/150 | Loss: 1538.8701 | Val Loss: 192.6266 | Val Qini: 0.9989 (patience: 5/20)EMA Trend: 1.0495 | (patience: 5/20)
Epoch 8/150 | Loss: -0.7485 | Val Loss: 192.5535 | Val Qini: 0.9757 (patience: 6/20)EMA Trend: 1.0385 | (patience: 

/home/ducm/Benchmark-conversion-vs-revenue-uplift-modeling/Revenue/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
